Kevin Hardegree-Ullman's homogeneous Kepler & K2 data products (https://github.com/kevinkhu/KeplerK2) from the latest Scaling K2 paper, https://iopscience.iop.org/article/10.3847/1538-3881/adf633#ajadf633f4 (Hardegree-Ullman+25, see Fig 4), should make my joint Kepler & K2 analysis more straightforward. The paper came out after we first submitted.

In [134]:
import os
import re
import os.path
import numpy as np
from numpy import log, exp, pi
import pandas as pd
import scipy
import random
from scipy.stats import gaussian_kde, loguniform, gamma
from math import lgamma
from tqdm import tqdm
from ast import literal_eval
from glob import glob
from tqdm import tqdm
from itertools import zip_longest
import numpy.ma as ma # for masked arrays
from astropy.table import Table, join
import astropy.coordinates as coord
import astropy.units as u
import gala.dynamics as gd
import gala.potential as gp
from pyia import GaiaData
from astropy.io import fits
from scipy.stats import ks_2samp
from scipy.stats import anderson_ksamp

# these packages are for fitting with numpyro
import numpyro
from numpyro import distributions as dist, infer
import numpyro_ext
import arviz as az
import jax

# these are psps imports
from psps.transit_class import Population, Star
import psps.simulate_helpers as simulate_helpers
import psps.simulate_transit as simulate_transit
import psps.utils as utils

# plotting imports
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
matplotlib.rcParams.update({'errorbar.capsize': 1})
pylab_params = {'legend.fontsize': 'large',
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large'}
pylab.rcParams.update(pylab_params)

import warnings
warnings.filterwarnings("ignore")

path = '/Users/chrislam/Desktop/psps/' 

# we're gonna need this for reading in the initial Berger+ 2020 data
def literal_eval_w_exceptions(x):
    try:
        return literal_eval(str(x))   
    except Exception as e:
        pass

def cull4d(df, teff, logg, feh, age, age_err1=None, age_err2=None):
    df = df.loc[(df[teff]<7500)&(df[teff]>3900)]
    df = df.loc[(df[logg]<4.7)&(df[logg]>4.)]
    df = df.loc[(df[feh]<0.25)&(df[feh]>-0.25)]
    df = df.loc[(df[age]<14)]
    """
    try:
        df['frac_age_err1'] = df[age_err1]/df[age]
        df['frac_age_err2'] = np.abs(df[age_err2]/df[age])
        print(np.nanmedian(df['frac_age_err1']), np.nanmedian(df['frac_age_err2']))
        #df = df.loc[(df['frac_age_err1']<0.46)&(df['frac_age_err2']<0.38)]
        df = df.loc[(df['frac_age_err1']<0.50)&(df['frac_age_err2']<0.40)]
    except:
        print("wasn't able to apply fractional age error cut")
        pass
    """

    return df
    

In [2]:
k2_stars = pd.read_csv(path+'data/joint/K2MainSequenceStars.csv')
k2_planets = pd.read_csv(path+'data/joint/K2Planets.csv')
kepler_stars = pd.read_csv(path+'data/joint/KeplerMainSequenceStars.csv')
kepler_planets = pd.read_csv(path+'data/joint/KeplerPlanets.csv')
print(list(k2_stars.columns))
print(list(k2_planets.columns))
print(list(kepler_stars.columns))
print(list(kepler_planets.columns))
print(len(np.unique(k2_stars['EPIC_ID'])))
print(len(np.unique(k2_planets['EPIC'])))
print(len(np.unique(kepler_stars['Kepler_ID'])))
print(len(np.unique(kepler_planets['KIC'])))

['EPIC_ID', 'Campaign', 'GaiaDR3', '2MASS', 'RAJ2000', 'e_RAJ2000', 'DEJ2000', 'e_DEJ2000', 'Dist', 'e_Dist', 'E_Dist', 'Teff', 'e_Teff', 'logg', 'e_logg', '[Fe/H]', 'e_[Fe/H]', 'Lum', 'e_Lum', 'Rad', 'e_Rad', 'Mass', 'e_Mass', 'RUWE', 'Gmag', 'e_Gmag', 'BPmag', 'e_BPmag', 'RPmag', 'e_RPmag', 'Jmag', 'e_Jmag', 'Hmag', 'e_Hmag', 'Kmag', 'e_Kmag', 'BP-RP', 'BP-G', 'G-RP', 'RP-J', 'J-H', 'H-K', 'extinctionJ', 'extinctionH', 'extinctionK', 'absJ', 'e_absJ', 'absH', 'e_absH', 'absK', 'e_absK', 'BC', 'e_BC', 'CDPP80', 'CDPP1', 'CDPP1_5', 'CDPP2', 'CDPP2_5', 'CDPP3', 'CDPP4', 'CDPP5', 'CDPP6', 'CDPP7', 'CDPP8', 'CDPP9', 'CDPP10', 'd_span']
['EPIC', 'Campaign', 'Candidate', 'Rp/Rs', 'e_Rp/Rs', 'Radius', 'e_Radius', 'Period', 'e_Period', 'a', 'e_a', 'Flux', 'e_Flux', 'CDPP', 'b']
['Kepler_ID', 'GaiaDR3', '2MASS', 'RAJ2000', 'e_RAJ2000', 'DEJ2000', 'e_DEJ2000', 'Dist', 'e_Dist', 'E_Dist', 'Teff', 'e_Teff', 'logg', 'e_logg', '[Fe/H]', 'e_[Fe/H]', 'Lum', 'e_Lum', 'Rad', 'e_Rad', 'Mass', 'e_Mass', 

Go from HU25 Kepler, minus binaries, minus non-astrometric solutions. Repeat for HU25, and combine. Then bootstrap to cull by Teff, logg, Fe/H, age. 

In [3]:
### Kepler
print(len(np.unique(kepler_stars['Kepler_ID'])))
# remove stars with RUWE < 1.2 (Penoyre+)
kepler_stars_keep = kepler_stars.loc[kepler_stars['RUWE']<1.2]
print(len(np.unique(kepler_stars_keep['Kepler_ID'])))

# remove stars without full astrometric solution
bedell = Table.read(path+'data/kepler_dr3_good.fits')
bedell_df = bedell.to_pandas()
kepler_stars_astrometry = pd.merge(kepler_stars_keep, bedell_df[['kepid', 'ra', 'dec', 'parallax', 'parallax_error', 'pmra', 'pmra_error', 'pmdec', 'pmdec_error', 'radial_velocity', 'radial_velocity_error']], left_on='Kepler_ID', right_on='kepid', how='left')
kepler_stars_astrometry = kepler_stars_astrometry.dropna(subset=['parallax', 'parallax_error', 'ra', 'dec', 'pmra', 'pmra_error', 'pmdec', 'pmdec_error', 'radial_velocity', 'radial_velocity_error'])
print("astrometric solution cut: ", len(np.unique(kepler_stars_astrometry['Kepler_ID'])))

### K2
print(len(np.unique(k2_stars['EPIC_ID'])))
# remove stars with RUWE < 1.2 (Penoyre+)
k2_stars_keep = k2_stars.loc[k2_stars['RUWE']<1.2]
print(len(np.unique(k2_stars_keep['EPIC_ID'])))



94422
90887
astrometric solution cut:  36054
90344
84459


In [4]:
# remove stars without full astrometric solution
from astroquery.gaia import Gaia
import pandas as pd

# list of Gaia DR3 source IDs
k2_source_ids = list(k2_stars_keep.GaiaDR3)

# convert to ADQL list
k2_source_list = ",".join(str(s) for s in k2_source_ids)

query = f"""
SELECT source_id,
       ra, dec,
       parallax,
       pmra, pmdec,
       radial_velocity,
	   ra_error, dec_error, parallax_error, pmra_error, pmdec_error, radial_velocity_error
FROM gaiadr3.gaia_source
WHERE source_id IN ({k2_source_list})
"""

job = Gaia.launch_job_async(query)
results = job.get_results()

k2_astrometry = results.to_pandas().dropna()

k2_stars_keep = pd.merge(k2_stars_keep, k2_astrometry, left_on='GaiaDR3', right_on='SOURCE_ID')
print(k2_stars_keep)
print(len(np.unique(k2_stars_keep['EPIC_ID'])))

INFO: Query finished. [astroquery.utils.tap.core]
         EPIC_ID  Campaign              GaiaDR3                    2MASS  \
0      201048855        10  3582456140266586240  2MASS J12075627-0809079   
1      201054542        10  3582466997945503616  2MASS J12085170-0756545   
2      201073427        10  3594616154755833984  2MASS J12092489-0721134   
3      201074674        10  3594605812474584704  2MASS J12085541-0719005   
4      201075635        10  3594620587162117632  2MASS J12103394-0717167   
...          ...       ...                  ...                      ...   
64130  251629442        17  3687041338408186368  2MASS J13214164-0032167   
64131  251630865        17  3686728806523722496  2MASS J13181428-0029166   
64132  251630989        17  3686684654259999360  2MASS J13195387-0029046   
64133  251632017        17  3686729906035351168  2MASS J13180690-0027037   
64134  251634367        17  3686732036339175552  2MASS J13183089-0022536   

          RAJ2000  e_RAJ2000   DEJ200

Crossmatch with B20 and B26 for ages


In [84]:
### Kepler
table2 = pd.read_csv(path+'data/GKSPCPapTable2_cleaned.txt', sep='&', header=0)

# remove stars with unreliable ages or bad goodness of fit
table2 = table2.loc[(table2.unReAgeFlag.isnull()) & (table2.iso_gof==1.)]
print("unreliable age cut: ", len(table2))

# merge
kepler = pd.merge(kepler_stars_astrometry, table2, left_on='Kepler_ID', right_on='KIC', how='inner')

# rename columns from Kepler to make them and K2 names unified
kepler['Age'] = kepler['iso_age']
kepler['E_Age'] = kepler['iso_age_err1']
kepler['e_Age'] = kepler['iso_age_err2']

### K2
b26 = pd.read_csv(path+'data/joint/berger2026.csv', sep='\s+') 
b26_k2 = b26[b26['ID'].str.startswith('epic')]
b26_k2['EPIC'] = b26_k2['ID'].str.strip('epic').astype(int)
print(b26_k2)
print(len(np.unique(b26_k2['EPIC'])))

# merge
k2 = pd.merge(k2_stars_keep, b26_k2, left_on='EPIC_ID', right_on='EPIC', how='inner')
k2['Teff'] = k2['Teff_x'] # HU25 and B26 have the same column names, so let's use HU25
k2['e_Teff'] = k2['e_Teff_x']
k2['logg'] = k2['logg_x']
k2['e_logg'] = k2['e_logg_x']

# combine kepler and k2
kepler_k2_keep = pd.concat([kepler, k2]).drop_duplicates(subset=['Kepler_ID','EPIC_ID'])
print(kepler_k2_keep)


unreliable age cut:  150560
                 ID  Teff  E_Teff  e_Teff   logg  E_logg  e_logg    feh  \
944   epic249822095  5513     275     304  3.963   0.100   0.115 -0.161   
1107  epic249239123  7302     235     236  4.242   0.046   0.054 -0.091   
1270  epic250062008  6845     249     223  3.804   0.056   0.058  0.003   
3128  epic215358983  6287     236     179  4.110   0.072   0.067 -0.008   
3171  epic205944181  5281     107     120  4.516   0.049   0.070 -0.079   
...             ...   ...     ...     ...    ...     ...     ...    ...   
4019  epic220491409  4420     103      96  4.606   0.036   0.041 -0.095   
4020  epic210894022  6213     206     202  4.320   0.072   0.081 -0.491   
4021  epic210910177  7213     269     255  4.136   0.059   0.061  0.050   
4022  epic220675519  5688     112     121  4.502   0.034   0.058 -0.147   
4023  epic249260568  3268      60      52  5.028   0.041   0.041 -0.250   

      E_feh  e_feh  ...  e_L_star  f_Age     Age   E_Age  e_Age       D

### Bootstrap/cull

In [103]:
def cull3d(df, teff, logg, feh):
    df = df.loc[(df[teff]<7500)&(df[teff]>3900)]
    print("teff cut: ", len(df))
    df = df.loc[(df[logg]<4.7)&(df[logg]>4.)]
    print("logg cut: ", len(df))
    df = df.loc[(df[feh]<0.25)&(df[feh]>-0.25)]
    print("feh cut: ", len(df))

    return df

def draw_age(support, mode_series, err1_series, err2_series):
    """

    """
    mode_series = mode_series.to_list()
    err1_series = err1_series.to_list()
    err2_series = np.abs(err2_series).to_list()
    draws = []
    for i in range(len(mode_series)):
        mode = mode_series[i]
        err1 = err1_series[i]
        err2 = err2_series[i]

        # symmetric uncertainties
        if err1==err2:
            draw = 0
            while draw <= 0: # make sure the draw is positive
                draw = np.around(np.random.normal(mode, err1), 2)

        # asymmetric uncertainties
        elif err1!=err2:
            pdf = simulate_helpers.make_pdf_rows(support, mode, err1, err2)
            pdf = pdf/np.sum(pdf)

            try:
                draw = 0
                while draw <= 0: # make sure the draw is positive
                    draw = np.around(np.random.choice(support, p=pdf), 2)
            except Exception as e:
                print("EXCEPTION: ", pdf, mode, err1, err2)
                print(e)
                break
        
        draws.append(draw)
        
    return draws

def gala_galactic_heights(df, output=True):


    """
    Use Gala (Price-Whelan+) to simulate orbits of Gaia stars and get their Z_maxes
    """

    """
    # merge sample with Megan Bedell's Gaia-Kepler cross-match because those save info on RV, proper motion, parallax, etc required for Gala
    berger = Table.read(path+'data/berger_kepler_stellar_fgk.csv')
    megan = Table.read(path+'data/kepler_dr3_good.fits')
    merged = join(berger, megan, keys='kepid')
    merged.rename_column('parallax_2', 'parallax')
    #print(merged[['parallax', 'parallax_error', 'radial_velocity', 'radial_velocity_error']])
    """

    df['radial_velocity'] = np.random.normal(df['radial_velocity'], df['radial_velocity_error'])
    df['pmra'] = np.random.normal(df['pmra'], df['pmra_error'])
    df['pmdec'] = np.random.normal(df['pmdec'], df['pmdec_error'])
    df['parallax'] = np.random.normal(df['parallax'], df['parallax_error'])

    # mise en place
    with coord.galactocentric_frame_defaults.set("v4.0"):
        galcen_frame = coord.Galactocentric()

    sun_xyz = u.Quantity(
        [-galcen_frame.galcen_distance, 0 * u.kpc, galcen_frame.z_sun]  # x  # y  # z
    )

    sun_w0 = gd.PhaseSpacePosition(pos=sun_xyz, vel=galcen_frame.galcen_v_sun)

    mw_potential = gp.MilkyWayPotential()

    sun_orbit = mw_potential.integrate_orbit(sun_w0, dt=0.5 * u.Myr, t1=0, t2=4 * u.Gyr)

    star_gaia = GaiaData(df)

    star_gaia_c = star_gaia.get_skycoord()
    star_galcen = star_gaia_c.transform_to(galcen_frame)
    star_w0 = gd.PhaseSpacePosition(star_galcen.data)

    # calculate orbits and retrieve Z_maxes
    zmaxes = []
    for i in tqdm(range(len(star_gaia))):
    #for i in range(1000):
        star_orbit = mw_potential.integrate_orbit(star_w0[i], t=sun_orbit.t) 
        zmax = star_orbit.zmax().value
        zmaxes.append(zmax)

    #zmaxes_df = pd.DataFrame({'height': zmaxes})
    #zmaxes_df.to_csv(path+'data/zmaxes.csv', index=False)
    
    return zmaxes

In [86]:
# bootstrap
kepler_k2_keep['teff_draw'] = np.random.normal(kepler_k2_keep['Teff'], kepler_k2_keep['e_Teff'])
kepler_k2_keep['logg_draw'] = np.random.normal(kepler_k2_keep['logg'], kepler_k2_keep['e_logg'])
kepler_k2_keep['feh_draw'] = np.random.normal(kepler_k2_keep['[Fe/H]'], kepler_k2_keep['e_[Fe/H]'])

# cull
kepler_k2_keep_bootstrap = cull3d(kepler_k2_keep, 'teff_draw', 'logg_draw', 'feh_draw')
print(len(kepler_k2_keep_bootstrap))


teff cut:  29973
logg cut:  29033
feh cut:  18316
18316


In [87]:
age_support = np.linspace(0.5, 13.5, 100)

kepler_k2_keep_bootstrap['age_draw'] = draw_age(age_support, kepler_k2_keep_bootstrap['Age'], kepler_k2_keep_bootstrap['E_Age'], kepler_k2_keep_bootstrap['e_Age'])    
kepler_k2_keep_bootstrap = kepler_k2_keep_bootstrap.loc[kepler_k2_keep_bootstrap['age_draw'] < 14.]
print(len(kepler_k2_keep_bootstrap))

18313


Good, it works. Now let's make 30 of these. 

In [108]:
len_teff = [] # track surviving star count so I can report on Table
len_logg = []
len_feh = []
len_age = []
for i in tqdm(range(30)):

    # deep copy so that changes here won't change the OG DF
    kepler_k2_keep_bootstrap = kepler_k2_keep.copy(deep=True) 
    
    # redraw
    kepler_k2_keep_bootstrap['teff_draw'] = np.random.normal(kepler_k2_keep_bootstrap['Teff'], kepler_k2_keep_bootstrap['e_Teff'])
    kepler_k2_keep_bootstrap['logg_draw'] = np.random.normal(kepler_k2_keep_bootstrap['logg'], kepler_k2_keep_bootstrap['e_logg'])
    kepler_k2_keep_bootstrap['feh_draw'] = np.random.normal(kepler_k2_keep_bootstrap['[Fe/H]'], kepler_k2_keep_bootstrap['e_[Fe/H]'])

    # re-cull
    kepler_k2_keep_bootstrap = kepler_k2_keep_bootstrap.loc[(kepler_k2_keep_bootstrap['teff_draw']<=7500)&(kepler_k2_keep_bootstrap['teff_draw']>=3900)]
    len_teff.append(len(kepler_k2_keep_bootstrap))
    #print("teff cut: ", len(kepler_k2_keep_bootstrap))

    kepler_k2_keep_bootstrap = kepler_k2_keep_bootstrap.loc[(kepler_k2_keep_bootstrap['logg_draw']<=4.7)&(kepler_k2_keep_bootstrap['logg_draw']>=4.)]
    len_logg.append(len(kepler_k2_keep_bootstrap))
    #print("logg cut: ", len(kepler_k2_keep_bootstrap))

    kepler_k2_keep_bootstrap = kepler_k2_keep_bootstrap.loc[(kepler_k2_keep_bootstrap['feh_draw']<=0.25)&(kepler_k2_keep_bootstrap['feh_draw']>=-0.25)]
    len_feh.append(len(kepler_k2_keep_bootstrap))
    #print("feh cut: ", len(kepler_k2_keep_bootstrap))

    kepler_k2_keep_bootstrap['age_draw'] = draw_age(age_support, kepler_k2_keep_bootstrap['Age'], kepler_k2_keep_bootstrap['E_Age'], kepler_k2_keep_bootstrap['e_Age'])    
    kepler_k2_keep_bootstrap = kepler_k2_keep_bootstrap.loc[kepler_k2_keep_bootstrap['age_draw'] < 14.]
    len_age.append(len(kepler_k2_keep_bootstrap))
    #print("age cut: ", len(kepler_k2_keep_bootstrap))

    # assign Zmax
    kepler_k2_keep_bootstrap['height'] = gala_galactic_heights(Table.from_pandas(kepler_k2_keep_bootstrap))

    kepler_k2_keep_bootstrap.to_csv(path+f'data/populations/pop_{i}.csv', index=False)

print("teff: ", np.mean(len_teff), np.std(len_teff))
print("logg: ", np.mean(len_logg), np.std(len_logg))
print("feh: ", np.mean(len_feh), np.std(len_feh))
print("age: ", np.mean(len_age), np.std(len_age))



100%|██████████| 13/13 [23:02<00:00, 106.34s/it]

teff:  29972.076923076922 3.099914104555367
logg:  29024.23076923077 18.43973070950477
feh:  18306.384615384617 68.68270240364349
age:  18303.69230769231 68.97757117675125


oops, I forgot about stellar radius!

In [118]:
kepler_k2_keep_bootstrap['stellar_radius'] = np.random.normal(kepler_k2_keep_bootstrap['Rad'], kepler_k2_keep_bootstrap['e_Rad'])
kepler_k2_keep_bootstrap['stellar_mass'] = np.random.normal(kepler_k2_keep_bootstrap['Mass'], kepler_k2_keep_bootstrap['e_Mass'])

So the bootstrap columns are age_draw, teff_draw, logg_draw, and feh_draw. And we have Zmax. Now we can assign planetary systems, based on the models. 

In [ ]:
threshold = 11
frac1 = 0.33
frac2 = 0.33

name_thresh = 11
name_f1 = 33
name_f2 = 33
name = 'step_'+str(name_thresh)+'_'+str(name_f1)+'_'+str(name_f2)
#name = 'monotonic_'+str(name_f1)+'_'+str(name_f2) 
#name = 'piecewise_'+str(name_thresh)+'_'+str(name_f1)+'_'+str(name_f2) 

period_grid = np.logspace(np.log10(1), np.log10(40), 10) # formerly up to 300 days, but that was for Lam+Ballard24. Zink+23 did 1-40 days.
radius_grid = np.linspace(1, 4, 10)

height_bins = np.logspace(2, 3, 6) # ah, so the above are the midpoints of the actual bins they used, I guess
height_bin_midpoints = 0.5 * (np.logspace(2,3,6)[1:] + np.logspace(2,3,6)[:-1])

pop = Population(kepler_k2_keep_bootstrap['age_draw'], threshold, frac1, frac2)
frac_hosts = pop.galactic_occurrence_step(threshold, frac1, frac2)
#frac_hosts = pop.galactic_occurrence_monotonic(frac1, frac2)
#frac_hosts = pop.galactic_occurrence_piecewise(frac1, frac2, threshold)
intact_fracs = scipy.stats.truncnorm.rvs(0, 1, loc=0.18, scale=0.1, size=len(kepler_k2_keep_bootstrap))  # np vs JAX bc of random key issues

alpha_se = np.random.normal(-1., 0.2)
alpha_sn = np.random.normal(-1.5, 0.1)

### generate planetary systems
"""
kepler_k2_keep_bootstrap['midplane'] = np.random.uniform(low=-np.pi/2, high=np.pi/2, size=len(kepler_k2_keep_bootstrap))
kepler_k2_keep_bootstrap['prob_intact'] = simulate_helpers.assign_status(frac_hosts, intact_fracs)
kepler_k2_keep_bootstrap['status'] = np.random.choice(['no-planet', 'intact', 'disrupted'], p=kepler_k2_keep_bootstrap['prob_intact'])
#self.sigma_incl = np.where(self.status=='intact', np.pi/90, np.pi/22.5) # numpy version of above
kepler_k2_keep_bootstrap['num_planets'] = simulate_helpers.assign_num_planets(kepler_k2_keep_bootstrap['status'])
fasdfadsf

### draw planet radii and periods such that they satisfy a Hill radius check and satisfy population statistics of radius valley. also draw planet masses.
self.periods, self.planet_radii, self.planet_masses = simulate_helpers.draw_planet_periods_and_radii(self.num_planets, self.alpha_se, self.alpha_sn, self.stellar_mass)
"""

# create Star objects, with their planetary systems
star_data = []
for i in tqdm(range(len(kepler_k2_keep_bootstrap))): 
    star = GeneralStar(kepler_k2_keep_bootstrap['GaiaDR3'][i], kepler_k2_keep_bootstrap['age_draw'][i], kepler_k2_keep_bootstrap['stellar_radius'][i], kepler_k2_keep_bootstrap['stellar_mass'][i])
    star_update = {
        'GaiaDR3': star.GaiaDR3,
        'age': star.age,
        'stellar_radius': star.stellar_radius,
        'stellar_mass': star.stellar_mass
    }
    asdfadsf
    
    star = Star(kepler_k2_keep_bootstrap['age_draw'][i], kepler_k2_keep_bootstrap['stellar_radius'][i], kepler_k2_keep_bootstrap['stellar_mass'][i], kepler_k2_keep_bootstrap['teff_draw'][i], kepler_k2_keep_bootstrap['CDPP6'][i], kepler_k2_keep_bootstrap['height'][i]*1000, alpha_se, alpha_sn, frac_hosts[i], intact_fracs[i], kepid=kepler_k2_keep_bootstrap['GaiaDR3'][i])
    star_update = {
        'kepid': star.kepid,
        'age': star.age,
        'stellar_radius': star.stellar_radius,
        'stellar_mass': star.stellar_mass,
        'Teff': star.Teff,
        'rrmscdpp06p0': star.rrmscdpp06p0,
        'frac_host': star.frac_host,
        'height': star.height,
        'midplane': star.midplane,
        'prob_intact': star.prob_intact, 
        'status': star.status,
        'sigma_incl': star.sigma_incl,
        'num_planets': star.num_planets,
        'periods': star.periods,
        'incls': star.incls,
        'mutual_incls': star.mutual_incls,
        'eccs': star.eccs,
        'omegas': star.omegas,
        'planet_radii': star.planet_radii
    }
    star_data.append(star_update)
    pop.add_child(star)

# convert back to DataFrame
kepler_k2_planets = pd.DataFrame.from_records(star_data)


  0%|          | 1/18329 [00:00<1:15:46,  4.03it/s]


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()